# Fake-News Detection Demo  
This notebook loads the pre-trained BERT fake-news classifier we trained earlier and lets you test it on custom input text.

In [61]:
import sys, subprocess, importlib.util
from transformers import BertTokenizer, BertModel, pipeline
import torch, numpy as np
from pathlib import Path
import joblib

def _ensure(pkg):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for p in ("joblib", "ipywidgets", "torch", "transformers", "pandas", "numpy"):
    _ensure(p)

In [62]:
# ---- load the pickles ----
PREPROC_DIR = Path("..") / "nlp" / "data" / "preprocessed"
fake_dict   = joblib.load(PREPROC_DIR / "bert_ner_model.pkl")
stance_dict = joblib.load(PREPROC_DIR / "bert_ner_stance_model.pkl")

pca_fake,  clf_fake  = fake_dict["pca"],  fake_dict["clf"]
pca_stan,  clf_stan  = stance_dict["pca"], stance_dict["clf"]

# ---- lightweight BERT & NER helpers ----
tok   = BertTokenizer.from_pretrained("bert-base-uncased")
bert  = BertModel.from_pretrained("bert-base-uncased"); bert.eval()
ner = pipeline(
    "ner",
    model="dbmdz/bert-base-cased-finetuned-conll03-english",
    tokenizer="dbmdz/bert-base-cased-finetuned-conll03-english",
    aggregation_strategy="simple",
    framework="pt",   # ← tell Transformers to use PyTorch only
    device=-1         # -1 = CPU   (0 would be first GPU)
)

Some weights of the model checkpoint at dbmdz/bert-base-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [63]:
def build_vec(text: str) -> np.ndarray:
    # CLS embedding
    with torch.no_grad():
        cls = bert(**tok(text, return_tensors="pt",
                         truncation=True, max_length=512)).last_hidden_state[0,0].numpy()
    # simple NER counts
    counts = dict.fromkeys(["PER","ORG","LOC","MISC"], 0)
    for e in ner(text[:512]):
        if e["entity_group"] in counts:
            counts[e["entity_group"]] += 1
    ner_vec = np.array([counts[c] for c in ("PER","ORG","LOC","MISC")], dtype=np.float32)
    return np.concatenate([cls, ner_vec])

In [64]:
stance_clf_hf = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    framework="pt",          # ← force PyTorch; avoids tf-keras issue
    device=-1                # -1 = CPU   (0 for first GPU)
)


def stance_flags(headline: str, body: str) -> np.ndarray:
       """
       Returns one-hot [contradicts, supports] for a head-vs-body pair.
       """
       res = stance_clf_hf(
           body,
           candidate_labels=["contradicts", "supports", "neutral"],
           hypothesis_template=f"The article {{}} the claim: '{headline}'."
       )["labels"][0]
       if res == "contradicts":
           return np.array([1, 0], dtype=np.float32)
       elif res == "supports":
           return np.array([0, 1], dtype=np.float32)
       else:                              # neutral
           return np.array([0, 0], dtype=np.float32)

Device set to use cpu


---

### Demo ‒ test the models

1. Edit **title** and **text** below.  
2. Run the cell to set the variables.  
3. Run the two prediction cells to see the base-model and stance-model verdicts.

---

In [65]:
# --- Edit this text then run the NEXT cell ---
title = "trump nominate Goldman Sachs Donovan deputy Treasury secretary"
text     = """
WASHINGTON Reuters President Donald Trump nominate Goldman Sachs banker James Donovan deputy Treasury secretary White House say Tuesday add another alumnus Wall Street investment bank administration Treasury Secretary Steven Mnuchin National Economic Council director Gary Cohn also former Goldman executive occupy senior economic post within administration Donovan work bank manage director include work corporate strategy investment banking investment management White House say statement he expect work Trump administration domestic policy agenda Treasury the White House also name David Malpass former official Ronald Reagan George Bush administration nominee Treasury undersecretary international affair key economic diplomacy post Malpass also serve former economist Wall Street bank Bear Stearns prior collapse recently serve economic adviser Trump campaign the White House also name former national security federal law enforcement official Sigal Mandelker Treasury top sanction post undersecretary terrorism financial intelligence a former law clerk Supreme Court Justice Clarence Thomas Mandelker later hold series criminal prosecution position Department Justice advise Secretary Homeland Security George Bush administration
"""

In [66]:
# build the vector for the input text
vec  = build_vec(text).reshape(1, -1)

In [67]:
# ==== prediction cell ====
vec       = build_vec(text).reshape(1, -1)
probs     = clf_fake.predict_proba(pca_fake.transform(vec))[0]
pred      = probs.argmax()          # 0 or 1
label     = "Fake" if pred else "Real"
conf      = probs[pred]

print(f"Base model → {label}  (confidence = {conf:.2%})")

Base model → Real  (confidence = 87.54%)


In [68]:
# ---- stance-augmented prediction ----
stance_feat = stance_flags(title, text).reshape(1, -1)      # (1, 2)
vec_full    = np.hstack([vec, stance_feat])                 # (1, 774)

probs_stan  = clf_stan.predict_proba(pca_stan.transform(vec_full))[0]  # [P(real), P(fake)]
pred_stan   = probs_stan.argmax()                           # 0 or 1
label_stan  = "Fake" if pred_stan else "Real"
conf_stan   = probs_stan[pred_stan]

print(f"Stance model → {label_stan}  (confidence = {conf_stan:.2%})")

Stance model → Real  (confidence = 76.92%)
